# Lesson 2.3: Combining Events: OR and AND

Check the addition rule (heart or king) and the multiplication rule for independent events (heads and a 6) by exact counting and by simulation.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. Build a deck and count

We number the 52 cards 0 to 51: the first 13 numbers are hearts, and the last card in each group of 13 is a king. Counting gives the pieces of the addition rule, including the king of hearts that is counted twice.

In [1]:
import numpy as np

cards = np.arange(52)
heart = cards // 13 == 0
king = cards % 13 == 12

print('hearts:', heart.sum(), ' kings:', king.sum(), ' both:', (heart & king).sum())
print('heart or king, counted directly:', (heart | king).sum(), 'of 52')

## 2. The addition rule

P(A or B) = P(A) + P(B) - P(A and B). Subtracting the overlap fixes the double count, and the answer matches the direct count.

In [2]:
p_heart = heart.mean()
p_king = king.mean()
p_both = (heart & king).mean()

print(f'P(heart) + P(king) = {p_heart + p_king:.4f}  (too big)')
print(f'minus overlap {p_both:.4f}   = {p_heart + p_king - p_both:.4f}')
print(f'direct count 16/52    = {(heart | king).mean():.4f}')

## 3. Check the addition rule by simulation

Now we really draw cards, one at a time with replacement, using a seeded generator. The share of draws that are a heart or a king should land close to 16/52.

In [3]:
rng = np.random.default_rng(42)
draws = rng.integers(0, 52, size=200_000)

sim_or = ((draws // 13 == 0) | (draws % 13 == 12)).mean()
print(f'simulated P(heart or king) = {sim_or:.4f}')
print(f'exact 16/52                = {16 / 52:.4f}')

## 4. The multiplication rule

A coin and a die do not affect each other, so the events are independent and P(heads and 6) = 1/2 x 1/6. We flip and roll 200,000 times and count how often both happen.

In [4]:
coin = rng.integers(0, 2, size=200_000)
die = rng.integers(1, 7, size=200_000)

both = ((coin == 1) & (die == 6)).mean()
print(f'simulated P(heads and 6) = {both:.4f}')
print(f'1/2 x 1/6                = {0.5 * 1 / 6:.4f}')
print(f'P(heads) x P(6) in sample = {(coin == 1).mean() * (die == 6).mean():.4f}')

## Try this

- Change the king to a queen of any suit and the heart to a spade. Does the overlap stay at 1 card?
- Roll a die twice in the simulation and ask for two sixes. Predict the answer (1/36) before running.

## The same idea in R, SPSS and Stata

**R**

```r
card <- 0:51
heart <- card %/% 13 == 0
king  <- card %% 13 == 12
mean(heart) + mean(king) - mean(heart & king)
mean(heart | king)            # direct count

0.5 * 1/6                      # heads and a 6

set.seed(42)
n <- 200000
mean(sample(0:1, n, TRUE) == 1 & sample(6, n, TRUE) == 6)
```

**SPSS** (Transform > Compute Variable; Analyze > Descriptive Statistics > Descriptives)

```
INPUT PROGRAM.
LOOP #i = 0 TO 51.
COMPUTE card = #i.
END CASE.
END LOOP.
END FILE.
END INPUT PROGRAM.
COMPUTE heart = (TRUNC(card/13) = 0).
COMPUTE king = (MOD(card, 13) = 12).
COMPUTE both = (heart = 1 AND king = 1).
COMPUTE either = (heart = 1 OR king = 1).
DESCRIPTIVES VARIABLES=heart king both either.
```

**Stata**

```
clear
set obs 52
gen card = _n - 1
gen heart = floor(card/13) == 0
gen king = mod(card, 13) == 12
gen either = heart | king
summarize heart king either
display 0.5 * 1/6

clear
set seed 42
set obs 200000
gen win = (runiform() < 0.5) & (ceil(6*runiform()) == 6)
summarize win
```